In [37]:
import librosa 
import librosa.display
import numpy as np
import pandas as pd
import matplotlib as plt
import seaborn as sns
import IPython.display as idp

from glob import glob
from itertools import cycle

In [38]:
path = '0audio/'
np.set_printoptions(threshold=np.inf, linewidth=np.inf)

In [39]:
def apply_vad(y, top_db, frame_length=551, hop_length=220):
    # Use librosa.effects.split to find non-silent intervals. This returns a 2xn array. which are the starting and stopping times of speech frames.
    intervals = librosa.effects.split(  y,                        
                                        top_db=top_db,                
                                        frame_length=frame_length,   
                                        hop_length=hop_length)     
    # Since some rhythym and natural pauses in speech can be used in speaker identification we only remove preceding and trailing silences.  
    start = intervals[0][0]
    end = intervals[-1][1]
    # Stitch the intervals back into one continuous speech-only array
    y_vad = np.concatenate([y[start:end]])
    return y_vad

In [40]:
def extract_fbanks(y_vad, sr, frame_length=551, hop_length=220):
    # Compute Mel spectrogram (power), then log-scale
    mel = librosa.feature.melspectrogram(
        y=y_vad,             # 1D speech‐only waveform
        sr=sr,              
        n_fft=frame_length,
        hop_length=hop_length,# setting the Frame size
        win_length=frame_length,# setting the Hop length
        window="hann",       # Kaldi’s default “hamming/povey” style → librosa’s “hann” is fine
        center=False,        # so frames align exactly with the 25 ms/10 ms grid
        n_mels=24,
        fmin=20,
        fmax=7600
    )
    log_fbank = librosa.power_to_db(mel, ref=np.max)  # shape: (24, n_frames)
    return log_fbank

In [41]:
def sliding_window_mean_normalization(log_fbank, window_size):
    # fbank: (n_mels, n_frames)
    df = pd.DataFrame(log_fbank.T)
    normalized = df.rolling(
        window=window_size,    # number of consecutive rows in each window
        min_periods=1,         # require at least 1 row to compute a mean
        center=True            # center the window on each row, rather than “trailing” or “leading”
    ).mean() # (axis=0)  # compute mean across rows in each window

    log_fbank_norm = (df - normalized).T.values.astype(np.float32) 
    return log_fbank_norm

In [ ]:
def process_audio_files(base_path):
    # Create an empty list to store the results
    results = []
    
    # Get all actor directories
    actor_directories = glob(base_path + "actor_*")
    
    # Process each actor folder
    for actor_folder in actor_directories:
        # Extract actor ID
        actor_id = actor_folder.split('\\')[-1].split('_')[1]
        
        # Get all WAV files in this actor folder
        audio_files = glob(actor_folder + "/*.wav")
        
        # Process each audio file
        for file in audio_files:
            try:
                # Extract filename
                filename = file.split('\\')[-1]
                
                # Load audio
                y, sr = librosa.load(file, sr=None)
                
                # define frame length and hop length
                # These values are based on the original Kaldi setup
                # Frame length is 25 ms, hop length is 10 ms
                frame_length = 551
                hop_length = 220

                # Apply VAD (Voice Activity Detection)
                # Using a top_db threshold to filter out silent parts
                y_vad = apply_vad(y, top_db=40, frame_length=frame_length, hop_length=hop_length)

                # Extract log Mel filter banks
                log_fbank = extract_fbanks(y_vad, sr, frame_length=frame_length, hop_length=hop_length)
                
                # Apply sliding window normalization
                windw_len_seconds = 3 # length of the sliding window in seconds
                # Calculate the window size in terms of frames
                window_size = int(windw_len_seconds * sr / hop_length)
                log_fbank_norm = sliding_window_mean_normalization(log_fbank, window_size=window_size)
                
                # Store results
                results.append({
                    'file': file,
                    'actor_id': actor_id,
                    'log_fbank_norm': log_fbank_norm
                })
                
                # Print progress
                if len(results) % 10 == 0:
                    print(f"Processed {len(results)} files...")
                    
            except Exception as e:
                print(f"Error processing file {file}: {str(e)}")
    
    # Convert to DataFrame
    df = pd.DataFrame(results)
    return df

# Process all audio files
audio_df = process_audio_files(path)
print(f"Processed {len(audio_df)} audio files")

Processed 10 files...
Processed 20 files...
Processed 30 files...
Processed 40 files...
Processed 50 files...
Processed 60 files...
Processed 70 files...
Processed 80 files...
Processed 90 files...
Processed 100 files...
Processed 110 files...
Processed 120 files...
Processed 130 files...
Processed 140 files...
Processed 150 files...
Processed 160 files...
Processed 170 files...
Processed 180 files...
Processed 190 files...
Processed 200 files...
Processed 210 files...
Processed 220 files...
Processed 230 files...
Processed 240 files...
Processed 250 files...
Processed 260 files...
Processed 270 files...
Processed 280 files...
Processed 290 files...
Processed 300 files...
Processed 310 files...
Processed 320 files...
Processed 330 files...
Processed 340 files...
Processed 350 files...
Processed 360 files...
Processed 370 files...
Processed 380 files...
Processed 390 files...
Processed 400 files...
Processed 410 files...
Processed 420 files...
Processed 430 files...
Processed 440 files.

# x-vector TDNN architecture

In [43]:
import tensorflow as tf
from tensorflow.keras import layers, Model
from sklearn.model_selection import train_test_split

In [ ]:
class XVectorTF(Model):
    def __init__(self, num_speakers: int):
        """
        Implements the x-vector TDNN as described in Snyder et al. (2018).
        
        Args:
          num_speakers: Number of target speakers for classification (e.g., 1191).
        """
        super(XVectorTF, self).__init__()
        # ---------------------------
        # 1) Frame-Level TDNN Layers
        # ---------------------------

        # Layer 1: Splice [-2,-1,0,+1,+2] on 24-dim log-Mel → 120 → 512
        # Implementation: Conv1D with kernel_size=5, dilation_rate=1, padding='same'
        self.tdnn1 = layers.Conv1D(
            filters=512,
            kernel_size=5,           # looks at 5 consecutive frames
            dilation_rate=1,         # no skipping
            padding='same',          # output length = input length
            use_bias=True           # use BatchNorm instead of bias
        )
        self.bn1 = layers.BatchNormalization()

        # Layer 2: Splice [-2,0,+2] on 512-dim → 1536 → 512
        # Implementation: kernel_size=3, dilation_rate=2 → covers frames t-2,t,t+2
        self.tdnn2 = layers.Conv1D(
            filters=512,
            kernel_size=3,
            dilation_rate=2,
            padding='same',
            use_bias=True
        )
        self.bn2 = layers.BatchNormalization()

        # Layer 3: Splice [-3,0,+3] on 512-dim → 1536 → 512
        # Implementation: kernel_size=3, dilation_rate=3 → covers t-3,t,t+3
        self.tdnn3 = layers.Conv1D(
            filters=512,
            kernel_size=3,
            dilation_rate=3,
            padding='same',
            use_bias=True
        )
        self.bn3 = layers.BatchNormalization()

        # Layer 4: Splice [0] on 512-dim → 512 → 512
        # Implementation: kernel_size=1, dilation_rate=1
        self.tdnn4 = layers.Conv1D(
            filters=512,
            kernel_size=1,
            dilation_rate=1,
            padding='valid',
            use_bias=True
        )
        self.bn4 = layers.BatchNormalization()

        # Layer 5: Splice [0] on 512-dim → 512 → 1500
        # Implementation: kernel_size=1, dilation_rate=1 → output dim = 1500
        self.tdnn5 = layers.Conv1D(
            filters=1500,
            kernel_size=1,
            dilation_rate=1,
            padding='valid',
            use_bias=True
        )
        self.bn5 = layers.BatchNormalization()

        # -------------------------------------
        # 2) Segment-Level (Dense) & Output
        # -------------------------------------

        # Layer 6: Stats-pooling output (3000-d) → 512
        self.segment6 = layers.Dense(
            units=512,
            use_bias=True
        )
        self.bn6 = layers.BatchNormalization()

        # Layer 7: 512 → 512
        self.segment7 = layers.Dense(
            units=512,
            use_bias=True
        )
        self.bn7 = layers.BatchNormalization()

        # Layer 8: 512 → num_speakers (logits)
        self.output_layer = layers.Dense(
            units=num_speakers,
            activation=None   # logits, softmax applied in loss
        )

    def forward(self, inputs, training=False):
        """
        Forward pass through the x-vector network.

        Args:
          inputs: Tensor of shape (batch_size, T, 24) = (N, T, 24 log-Mel fbanks).
          training: bool, whether in training mode (affects BatchNorm).

        Returns:
          logits: Tensor of shape (N, num_speakers) – raw classification scores.
          embeddings: Tensor of shape (N, 512) – the x-vector embedding (output of Layer 6).
        """
        x = inputs   # (N, T, 24)

        # ---- Frame-level TDNN ----
        # Layer 1: Conv1D 24→512, kernel=5, dilation=1, padding='same'
        h = self.tdnn1(x)                 # (N, T, 512)
        h = self.bn1(h, training=training)
        h = tf.nn.relu(h)                 # (N, T, 512)

        # Layer 2: Conv1D 512→512, kernel=3, dilation=2, padding='same'
        h = self.tdnn2(h)                 # (N, T, 512)
        h = self.bn2(h, training=training)
        h = tf.nn.relu(h)                 # (N, T, 512)

        # Layer 3: Conv1D 512→512, kernel=3, dilation=3, padding='same'
        h = self.tdnn3(h)                 # (N, T, 512)
        h = self.bn3(h, training=training)
        h = tf.nn.relu(h)                 # (N, T, 512)

        # Layer 4: Conv1D 512→512, kernel=1, dilation=1, padding='valid'
        h = self.tdnn4(h)                 # (N, T, 512)
        h = self.bn4(h, training=training)
        h = tf.nn.relu(h)                 # (N, T, 512)

        # Layer 5: Conv1D 512→1500, kernel=1, dilation=1, padding='valid'
        h = self.tdnn5(h)                 # (N, T, 1500)
        h = self.bn5(h, training=training)
        h = tf.nn.relu(h)                 # (N, T, 1500)

        # ---- Stats-Pooling (mean + std over time) ----
        # h has shape (N, T, 1500)
        mean = tf.reduce_mean(h, axis=1)                              # (N, 1500)
        var = tf.reduce_mean(tf.square(h - tf.expand_dims(mean,1)), axis=1)
        std = tf.sqrt(var + 1e-12)                                    # (N, 1500)
        stats = tf.concat([mean, std], axis=1)                        # (N, 3000)

        # ---- Segment-Level Layers ----
        # Layer 6: Dense(3000→512) + BN + ReLU → x-vector
        s = self.segment6(stats)              # (N, 512)
        s = self.bn6(s, training=training)
        s = tf.nn.relu(s)
        embeddings = tf.identity(s)           # (N, 512)

        # Layer 7: Dense(512→512) + BN + ReLU
        s = self.segment7(s)                  # (N, 512)
        s = self.bn7(s, training=training)
        s = tf.nn.relu(s)

        # Layer 8: Dense(512→num_speakers) → logits
        logits = self.output_layer(s)         # (N, num_speakers)

        return logits, embeddings

In [45]:
# 1) Instantiate model with 24 speakers as present int he RAVDESS dataset (see 0audio/LicenseAttribution.txt)
model = XVectorTF(num_speakers=24)

In [46]:
# 2) Add a column of integer labels
audio_df['actor_id'] = audio_df['actor_id'].astype(int)  # Ensure actor_id is int

In [47]:
train_df, test_df = train_test_split(
    audio_df,
    test_size=0.2,
    stratify=audio_df['actor_id'],
    random_state=42
)
print(f"Train size: {len(train_df)}, Test size: {len(test_df)}")

Train size: 1152, Test size: 288


In [54]:
segment_frames = 300    # 3 seconds @ 10 ms/frame

def generator_from_df(df):
    """Yields (features, label) pairs from a DataFrame."""
    for _, row in df.iterrows():
        # row['log_fbank_norm'] is shape (24, T)
        feat = row['log_fbank_norm'].T.astype(np.float32)  # (T,24)
        label = np.int32(row['actor_id'] - 1)
        yield feat, label

def preprocess_train(feat, label):
    """
    Randomly crop or pad `feat` to exactly `segment_frames`.
    feat: (T,24), tf.float32
    label: scalar tf.int32
    """
    T = tf.shape(feat)[0]
    # If longer than segment_frames, pick a random start
    def _crop():
        start = tf.random.uniform([], 0, T - segment_frames + 1, dtype=tf.int32)
        return feat[start:start+segment_frames]
    # If shorter, pad at end
    def _pad():
        pad_len = segment_frames - T
        return tf.pad(feat, [[0, pad_len],[0,0]])
    feat2 = tf.cond(T >= segment_frames, _crop, _pad)
    return feat2, label

def preprocess_test(feat, label):
    """
    Deterministically pad or truncate to segment_frames for validation.
    """
    T = tf.shape(feat)[0]
    feat2 = feat[:segment_frames]                             # truncate if T>segment_frames
    pad_len = tf.maximum(0, segment_frames - tf.shape(feat2)[0])
    feat2 = tf.pad(feat2, [[0, pad_len],[0,0]])               # pad if T<segment_frames
    return feat2, label

# Build datasets
train_ds = tf.data.Dataset.from_generator(
    lambda: generator_from_df(train_df),
    output_signature=(
        tf.TensorSpec(shape=(None, 24), dtype=tf.float32),
        tf.TensorSpec(shape=(),      dtype=tf.int32)
    )
).map(preprocess_train, num_parallel_calls=tf.data.AUTOTUNE) \
 .shuffle(buffer_size=1000) \
 .batch(32, drop_remainder=True) \
 .prefetch(tf.data.AUTOTUNE)

test_ds = tf.data.Dataset.from_generator(
    lambda: generator_from_df(test_df),
    output_signature=(
        tf.TensorSpec(shape=(None, 24), dtype=tf.float32),
        tf.TensorSpec(shape=(),      dtype=tf.int32)
    )
).map(preprocess_test, num_parallel_calls=tf.data.AUTOTUNE) \
 .batch(32) \
 .prefetch(tf.data.AUTOTUNE)

In [60]:
# 1) Instantiate (or reload) your model
model = XVectorTF(num_speakers=24)

# 2) Compile with optimizer + loss + metric
model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=1e-3),
    loss=tf.keras.losses.SparseCategoricalCrossentropy(from_logits=True),
    metrics=['accuracy']
)
# Debug: inspect one batch from train_ds
for x_batch, y_batch in train_ds.take(1):
    print("Feature batch shape:", x_batch.shape, "dtype:", x_batch.dtype)
    print("Label batch shape:  ", y_batch.shape, "dtype:", y_batch.dtype)
    # Try a forward pass
    logits, embeds = model(x_batch, training=True)
    print("→ Logits shape:", logits.shape, "Embeddings shape:", embeds.shape)
    break
# 3) Fit for, say, 10 epochs
history = model.fit(
    train_ds,
    validation_data=test_ds,
    epochs=10
)

# 4) Final evaluation
eval = model.evaluate(test_ds)

Feature batch shape: (32, 300, 24) dtype: <dtype: 'float32'>
Label batch shape:   (32,) dtype: <dtype: 'int32'>
→ Logits shape: (32, 24) Embeddings shape: (32, 512)
Epoch 1/10
36/36 [==============================] - 22s 552ms/step - loss: 8.0505 - output_1_loss: 2.4297 - output_2_loss: 5.6208 - output_1_accuracy: 0.3082 - output_2_accuracy: 0.1068 - val_loss: 20.5752 - val_output_1_loss: 11.3008 - val_output_2_loss: 9.2744 - val_output_1_accuracy: 0.0417 - val_output_2_accuracy: 0.0000e+00
Epoch 2/10
36/36 [==============================] - 22s 614ms/step - loss: 5.5064 - output_1_loss: 1.0046 - output_2_loss: 4.5017 - output_1_accuracy: 0.6962 - output_2_accuracy: 0.4661 - val_loss: 8.6442 - val_output_1_loss: 2.8844 - val_output_2_loss: 5.7599 - val_output_1_accuracy: 0.1667 - val_output_2_accuracy: 0.0799
Epoch 3/10
36/36 [==============================] - 24s 672ms/step - loss: 4.3073 - output_1_loss: 0.4400 - output_2_loss: 3.8673 - output_1_accuracy: 0.8663 - output_2_accuracy: 

In [62]:
print(eval)

[4.085992336273193, 1.1287275552749634, 2.9572641849517822, 0.6111111044883728, 0.4409722089767456]


In [75]:
# Create a function to extract x-vector embeddings from audio files
def extract_xvectors(df):
    # Create a dictionary to store speaker ID and corresponding x-vectors
    speaker_embeddings = {}
    
    # Process each file in the dataframe
    for idx, row in df.iterrows():
        # Get the feature and prepare it for the model
        feat = row['log_fbank_norm'].T.astype(np.float32)  # (T,24)
        
        # Pad or truncate to segment_frames
        if feat.shape[0] >= segment_frames:
            # If longer, take the first segment_frames frames
            feat = feat[:segment_frames]
        else:
            # If shorter, pad with zeros
            pad_len = segment_frames - feat.shape[0]
            feat = np.pad(feat, ((0, pad_len), (0, 0)))
        
        # Add batch dimension
        feat = np.expand_dims(feat, 0)  # Shape: (1, segment_frames, 24)
        
        # Extract embedding using the model
        _, embedding = model(feat, training=False)
        
        # Store the embedding with speaker ID
        actor_id = row['actor_id']
        if actor_id not in speaker_embeddings:
            speaker_embeddings[actor_id] = []
        speaker_embeddings[actor_id].append(embedding.numpy().flatten())
    
    # Convert to a DataFrame for easier analysis
    result = []
    for speaker_id, embeddings in speaker_embeddings.items():
        for emb in embeddings:
            result.append({
                'speaker_id': speaker_id,
                'embedding': emb
            })
    
    return pd.DataFrame(result)

# Extract x-vectors from test set
xvectors_df = extract_xvectors(test_df)

# Display sample results
print(f"Extracted {len(xvectors_df)} x-vectors")
print(f"Embedding dimension: {len(xvectors_df.iloc[0]['embedding'])}")
print(f"Number of unique speakers: {xvectors_df['speaker_id'].nunique()}")

Extracted 288 x-vectors
Embedding dimension: 512
Number of unique speakers: 24


In [76]:

xvectors_df

,speaker_id,embedding
0,21,"[0.0, 0.17301744, 1.5835581, 0.0, 0.65437436, ..."
1,21,"[1.7732921, 0.17154512, 0.0, 0.0, 0.35371172, ..."
2,21,"[3.0391552, 0.0, 0.0, 0.0, 0.2379173, 0.751002..."
3,21,"[0.0, 0.17114544, 1.2831109, 0.0, 1.2587578, 1..."
4,21,"[1.5384421, 0.0, 0.0, 0.0, 0.0, 0.82293224, 0...."
...,...,...
283,10,"[0.19825912, 0.0, 0.0, 0.29753327, 2.0022562, ..."
284,10,"[0.1133385, 0.0, 0.0, 0.0, 0.6173796, 0.737183..."
285,10,"[2.169364, 0.0, 0.0, 0.673753, 1.3998497, 1.00..."
286,10,"[0.3230055, 0.0, 0.0, 0.8982823, 0.4266867, 1...."


In [74]:
xvectors.shape

(1440, 512)